In [41]:
import os
import pickle
import torch
from torch.utils.data import Dataset
import random

AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20, 'X': 20  # padding 또는 unknown
}

def random_voxel_rotate(voxel):
    # voxel: Tensor [C, D, H, W]
    if random.random() < 0.5:  # 50% 확률로 회전 적용
        axes = [(2, 3), (1, 3), (1, 2)]  # (H, W), (D, W), (D, H)
        k = random.choice([1, 2, 3])  # 실제 회전만 (0 제외)
        axis = random.choice(axes)
        voxel = torch.rot90(voxel, k=k, dims=axis)
    return voxel

def random_voxel_flip(voxel):
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[1])  # D-axis flip
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[2])  # H-axis flip
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[3])  # W-axis flip
    return voxel

class VoxelDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, aug=False):
        self.df = df.reset_index(drop=True)
        self.voxel_cache_dir = voxel_cache_dir
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["PDB"]
        mut_pos = row["MAPPED_PDB_POS"]
        wt = row["WT"]
        mut = row["MT"]
        label = row["DDG"]

        key = f"{uid}_{mut_pos}"
        voxel_path = os.path.join(self.voxel_cache_dir, f"{key}.pkl")

        # Load voxel
        with open(voxel_path, "rb") as f:
            data = pickle.load(f)
            feature = data["feature"]  # shape: (1, 7, 7, 7, 63)

        # Preprocess
        feature_tensor = torch.from_numpy(feature).permute(0, 4, 1, 2, 3).float().squeeze(0)  # (63, 7, 7, 7)

        if self.aug:
            feature_tensor = random_voxel_rotate(feature_tensor)
            feature_tensor = random_voxel_flip(feature_tensor)

        # Convert WT/Mut AA to index
        ref_idx = torch.tensor(AA_TO_INDEX.get(str(wt), 20), dtype=torch.long)
        mut_idx = torch.tensor(AA_TO_INDEX.get(str(mut), 20), dtype=torch.long)

        return feature_tensor, ref_idx, mut_idx, torch.tensor(label).float()
    
class MSADataset(Dataset):
    def __init__(self, df, msa_dict_path, max_depth=80, win_size=61, aug=False, symmetry=False):
        with open(msa_dict_path, "rb") as f:
            self.msa_dict = pickle.load(f)
        self.df = df
        self.max_depth = max_depth
        self.win_size = win_size
        self.half_win = win_size // 2  # 중심에서 양쪽 길이
        self.aug = aug
        self.symmetry = symmetry
        
    def __len__(self):
        return len(self.df) * 2 if self.symmetry else len(self.df)

    def __getitem__(self, idx):

        if self.symmetry:
            real_idx = idx // 2
            is_reverse = idx % 2  # 0이면 정방향, 1이면 역방향
        else:
            real_idx = idx
            is_reverse = 0

        row = self.df.iloc[real_idx]
        uid = row["PDB"]
        mut_pos = int(row["POS"]) - 1  # 1-based → 0-based
        mut = row["MT"].upper()
        label = row["DDG"]

        msa_seqs = [seq for _, seq in self.msa_dict[uid][:self.max_depth]]
        query_seq = msa_seqs[0]  # 보통 첫 줄이 ref seq

        # 변이 반영된 mut_seq 생성
        mut_seq = list(query_seq)
        if 0 <= mut_pos < len(mut_seq):
            mut_seq[mut_pos] = mut

        # --- 핵심 로직: 1, 2행 구성 ---
        if is_reverse == 0:
            # 정방향 (Mut -> WT): 기존 방식
            seqs_to_use = [mut_seq, list(query_seq)]
            target_label = label
        else:
            # 역방향 (WT -> Mut): 대칭 증강
            seqs_to_use = [list(query_seq), mut_seq]
            target_label = -label  # 라벨 부호 반전
            
        if len(msa_seqs) > 1:
            seqs_to_use += [list(seq) for seq in msa_seqs[1:self.max_depth - 2]]

        if self.aug:
            msa_part = seqs_to_use[2:]  # 변이+ref 제외
            random.shuffle(msa_part)   # 순서 섞기
            seqs_to_use = seqs_to_use[:2] + msa_part

        centered_msa = []
        for seq in seqs_to_use:
            window = []
            for i in range(self.win_size):
                seq_idx = mut_pos - self.half_win + i
                if 0 <= seq_idx < len(seq):
                    aa = seq[seq_idx]
                else:
                    aa = '-'
                window.append(AA_TO_INDEX.get(aa, 20))
            centered_msa.append(window)

        # depth padding
        while len(centered_msa) < self.max_depth:
            centered_msa.append([20] * self.win_size)  # 20은 패딩 인덱스

        msa_tensor = torch.tensor(centered_msa[:self.max_depth]).long()  # [D, L]
        msa_tensor = msa_tensor.transpose(0, 1)  # [L, D]

        return {
            "msa": msa_tensor,  # [L, D]
            "label": torch.tensor(target_label).float()
        }

class MultimodalDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, msa_dict_path, 
                 voxel_aug=False, msa_aug=False, max_depth=80, win_size=61, symmetry=False):
        self.df = df.reset_index(drop=True)
        self.voxel_dataset = VoxelDataset(df, voxel_cache_dir, aug=voxel_aug)
        self.msa_dataset = MSADataset(df, msa_dict_path, max_depth=max_depth, win_size=win_size, aug=msa_aug, symmetry=symmetry)
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        voxel_feat, ref_idx, mut_idx, label = self.voxel_dataset[idx]
        msa_data = self.msa_dataset[idx]  # returns dict with "msa", "label"
        msa_tensor = msa_data["msa"]
        
        # 라벨 일치 확인 (안전용)
        assert label == msa_data["label"], "Mismatch in label!"

        return {
            "voxel": voxel_feat,      # [63, 7, 7, 7]
            "ref_idx": ref_idx,
            "mut_idx": mut_idx,
            "msa": msa_tensor,        # [L=61, D]
            "label": label
        }

In [42]:
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from Models import EvoStructCLIP
import torch

In [ ]:
import torch
import torch.nn as nn

class FeatureAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        # 3개의 특징(Global, Mut, WT) 사이의 관계를 파악하는 Attention
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.SiLU(),
            nn.Linear(dim * 2, dim)
        )
        self.norm_f = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, 3, 128] (3은 Global, Mut, WT)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm_f(x + ffn_out)
        return x

class StabilityRegressor(nn.Module):
    def __init__(self, clip_model_path, voxel_ch=46, embed_dim=128, unfreeze_msa=False):
        super().__init__()
        
        # 1. 사전 학습된 EvoStructCLIP 로드
        base_model = EvoStructCLIP(voxel_ch=voxel_ch, mb_layers=6, embed_dim=embed_dim)
        ckpt = torch.load(clip_model_path, map_location="cpu")
        base_model.load_state_dict(ckpt, strict=True)
        
        # 각 브랜치 추출
        self.msa_branch = base_model.msa_encoder 
        self.voxel_branch = base_model.voxel_encoder
        
        # 가중치 고정 설정 (필요에 따라 MSA만 미세조정 가능)
        for param in self.voxel_branch.parameters():
            param.requires_grad = False # 구조 정보는 Fine-tuning이 유리함
        
        if not unfreeze_msa:
            for param in self.msa_branch.parameters():
                param.requires_grad = False
        
        self.raw_norm = nn.LayerNorm(embed_dim)
        
        # 2. 통합 특징 Attention 레이어
        # 4개의 정보(Global MSA, Mut Seq, WT Seq, Voxel Structure)를 통합
        self.feature_attn = FeatureAttentionBlock(embed_dim)
        
        # 3. 최종 회귀 헤드 (입력 차원이 4 * embed_dim으로 확장됨)
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim * 3, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Linear(512, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )

    def forward(self, msa, voxel, ref_idx, mut_idx):
        # --- (A) MSA Branch 특징 추출 ---
        # MSA 정보는 고정된 상태에서 특징만 뽑을 경우 torch.no_grad() 적용 가능
        # 여기서는 전체 흐름을 위해 일반 forward로 진행
        x_msa = self.msa_branch.encoder(msa)
        B, L, D, C = x_msa.shape
        center_L = L // 2
        
        # Center-aware pooling 과정 재현
        query = x_msa[:, center_L].reshape(B * D, 1, C)
        keyval = x_msa.permute(0, 2, 1, 3).reshape(B * D, L, C)
        attn_out, _ = self.msa_branch.pooling.attn(query, keyval, keyval)
        per_seq_feats = attn_out.view(B, D, C)
        
        global_msa_feat = self.msa_branch.refine(per_seq_feats.mean(dim=1))
        mut_seq_feat = self.raw_norm(per_seq_feats[:, 0, :]) # Mutated sequence
        wt_seq_feat = self.raw_norm(per_seq_feats[:, 1, :])  # Wild-type sequence
        
        # --- (B) Voxel Branch 특징 추출 ---
        # 3D 구조 및 주변 원자 환경 정보
        voxel_feat = self.voxel_branch(voxel, ref_idx, mut_idx) # [B, 128]
        
        diff_feat = mut_seq_feat - wt_seq_feat

        # --- (C) 특징 통합 및 Attention ---
        # [B, 4, 128] 형태로 스택 (Global, Mut, WT, Structure)
        features = torch.stack([
            global_msa_feat, 
            mut_seq_feat, 
            wt_seq_feat, 
            diff_feat,
            voxel_feat
        ], dim=1) 
        
        # 4개 특징 간의 상호교차 검증 (Attention)
        features = self.feature_attn(features) # [B, 4, 128]
        
        # 최종 결합 및 회귀
        combined = features.reshape(B, -1) # [B, 512]
        return self.regressor(combined).squeeze(-1)

In [44]:
import torch.optim as optim
from scipy.stats import pearsonr
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

def pearson_loss(x, y):
    mx = torch.mean(x)
    my = torch.mean(y)
    xm, ym = x - mx, y - my
    r_num = torch.sum(xm * ym)
    r_den = torch.sqrt(torch.sum(xm**2) * torch.sum(ym**2) + 1e-8)
    r = r_num / r_den
    return 1 - r  # 상관계수가 1에 가까울수록 Loss는 0

def train_one_fold(fold_idx, train_df, val_df, msa_dict_path, voxel_cache_dir, clip_weight_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. 데이터로더 구성 (학습 효율을 위해 num_workers 설정)
    train_ds = MultimodalDataset(train_df, voxel_cache_dir, msa_dict_path, voxel_aug=False, msa_aug=False, symmetry=False)
    val_ds   = MultimodalDataset(val_df, voxel_cache_dir, msa_dict_path)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=8, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)
    
    
    # 2. 모델 및 최적화 설정
    model = StabilityRegressor(clip_weight_path).to(device)
    criterion = nn.HuberLoss(delta=1.0) # MSE와 MAE의 장점을 합친 Loss
    optimizer = optim.AdamW(model.regressor.parameters(), lr=1e-4, weight_decay=0.05) # 초기 LR 1e-3
    
    # 3. Cosine Annealing Scheduler 추가
    # T_max는 반주기 에폭 수입니다. 보통 전체 에폭 수와 같게 설정하여 끝까지 서서히 줄어들게 합니다.
    epochs = 10000
    scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)
    
    # 4. Early Stopping 설정
    patience = 500 # 15 에폭 동안 Pearson 점수가 오르지 않으면 중단
    counter = 0
    best_pearson = -1.0
    

    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        total_loss = 0
        for batch in train_loader:
            msa = batch["msa"].to(device)
            labels = batch["label"].to(device)
            voxel = batch["voxel"].to(device)
            ref_idx = batch["ref_idx"].to(device)
            mut_idx = batch["mut_idx"].to(device)
            
            optimizer.zero_grad()
            preds = model(msa, voxel, ref_idx, mut_idx)
            loss = criterion(preds, labels) + 0.5 * pearson_loss(preds, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # --- Validation Phase ---
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                msa = batch["msa"].to(device)
                labels = batch["label"].to(device)
                voxel = batch["voxel"].to(device)
                ref_idx = batch["ref_idx"].to(device)
                mut_idx = batch["mut_idx"].to(device)

                preds = model(msa, voxel, ref_idx, mut_idx) 
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # Pearson 상관계수 계산
        fold_pearson, _ = pearsonr(all_labels, all_preds)
        
        # 스케줄러 스텝 (Cosine Decay 적용)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # --- Early Stopping & Model Save Logic ---
        if fold_pearson > best_pearson:
            best_pearson = fold_pearson
            torch.save(model.state_dict(), f"best_reg_fold_{fold_idx}.pth")
            counter = 0 # 개선되었으므로 카운터 리셋
        else:
            counter += 1 # 개선되지 않음
            
        print(f"Fold {fold_idx} | Ep {epoch+1:3d}/{epochs} | LR: {current_lr:.6f} | Loss: {total_loss/len(train_loader):.4f} | Pearson: {fold_pearson:.4f} | Best: {best_pearson:.4f}")
        
        if counter >= patience:
            print(f"!!! Early Stopping triggered at epoch {epoch+1} !!!")
            break

    del model, optimizer, scheduler, train_loader, val_loader
    return best_pearson

In [45]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"/mnt/c/Users/Kunny/Documents/GitHub/CAGI/EvoStructCLIP/Stability/S2450_Final_PDB_Mapped.tsv", sep="\t", )

# 경로 설정
msa_dict_path = "/mnt/e/CAGI_data/msa_dict_S2450.pkl"
voxel_cache_dir = "/mnt/e/CAGI_data/voxel_cache_S2450"
clip_weight_path = "/mnt/e/CAGI_data/best_model_250917_clip_epoch81.pth"
fold_list = sorted(df['CVFOLD'].unique())

final_results = []

for f_idx in fold_list:
    print(f"\n--- Starting 5-Fold Cross Validation: Fold {f_idx} ---")
    val_df = df[df['CVFOLD'] == f_idx].copy()
    train_df = df[df['CVFOLD'] != f_idx].copy()
    
    best_score = train_one_fold(f_idx, train_df, val_df, msa_dict_path, voxel_cache_dir, clip_weight_path)
    final_results.append(best_score)

print("\n" + "="*40)
print(f"5-Fold CV Average Pearson Correlation: {np.mean(final_results):.4f}")
print("="*40)


--- Starting 5-Fold Cross Validation: Fold 0 ---
Fold 0 | Ep   1/10000 | LR: 0.000100 | Loss: 1.2407 | Pearson: 0.4156 | Best: 0.4156
Fold 0 | Ep   2/10000 | LR: 0.000100 | Loss: 0.9590 | Pearson: 0.4422 | Best: 0.4422
Fold 0 | Ep   3/10000 | LR: 0.000100 | Loss: 0.8617 | Pearson: 0.4421 | Best: 0.4422
Fold 0 | Ep   4/10000 | LR: 0.000100 | Loss: 0.8004 | Pearson: 0.4391 | Best: 0.4422
Fold 0 | Ep   5/10000 | LR: 0.000099 | Loss: 0.7541 | Pearson: 0.4481 | Best: 0.4481
Fold 0 | Ep   6/10000 | LR: 0.000099 | Loss: 0.7399 | Pearson: 0.4453 | Best: 0.4481
Fold 0 | Ep   7/10000 | LR: 0.000099 | Loss: 0.7050 | Pearson: 0.4580 | Best: 0.4580
Fold 0 | Ep   8/10000 | LR: 0.000098 | Loss: 0.6976 | Pearson: 0.4525 | Best: 0.4580
Fold 0 | Ep   9/10000 | LR: 0.000098 | Loss: 0.6988 | Pearson: 0.4592 | Best: 0.4592
Fold 0 | Ep  10/10000 | LR: 0.000098 | Loss: 0.6700 | Pearson: 0.4461 | Best: 0.4592
Fold 0 | Ep  11/10000 | LR: 0.000097 | Loss: 0.6611 | Pearson: 0.4458 | Best: 0.4592
Fold 0 | Ep  12

KeyboardInterrupt: 